In [ ]:
import os
from mongodb_client import MongoDBClient
mongo_uri = os.getenv("MONGO_URI")
db_name = os.getenv("DB_NAME")
collection = os.getenv("COLLECTION_NAME")
mongo_uri_source = os.getenv("MONGO_URI_SOURCE")
db_name_source = os.getenv("DB_NAME_SOURCE")
collection_source = os.getenv("COLLECTION_NAME_SOURCE")

In [ ]:
client = MongoDBClient(mongo_uri, db_name, collection)
client_source = MongoDBClient(mongo_uri_source, db_name_source, collection_source)

In [ ]:
client.connect()
client_source.connect()

In [ ]:
client.create_unique_index("url")

In [ ]:
sources = ["elpais", "opinion", "lostiempos", "eldiario", "larazon", "eldeber"]
total = 0
for source in sources:
    pipeline = [
        {
            '$match': {
                'source': source, 
                'tags': {
                    '$elemMatch': {
                        '$regex': 'feminicid', 
                        '$options': 'i'
                    }
                }
            }
        }
    ]
    documents = client_source.aggregate_documents(pipeline)
    if len(documents) == 0:
        pipeline = [
            {
                '$match': {
                    'source': source, 
                    'section': {
                        '$regex': 'seguridad'
                    }
                }
            }
        ]
        documents = client_source.aggregate_documents(pipeline)
    if len(documents) > 0:
        documents.drop(columns=["_id"])
    for doc in documents.to_dict(orient='records'):
        client.insert_new_document(doc, "url")
